# Example I: Neural Network for Handwritten Digit Classification

This notebook implements a three-layer feedforward neural network trained from scratch
to classify handwritten digits from the **MNIST** dataset.

## Architecture
```
Input layer   : 784 units  (one per pixel of the 28×28 image)
Hidden layer  :  30 units  (sigmoid activation)
Output layer  :  10 units  (sigmoid activation, one per digit class 0–9)
```

## Training
- **Loss:** Mean Squared Error (MSE) between the 10-dim network output and the one-hot target
- **Optimizer:** Mini-batch Stochastic Gradient Descent (SGD) with backpropagation
- **Hyperparameters:** learning rate = 3.0, mini-batch size = 10, epochs = 30

## Data splits
| Split      | Size   | Purpose                                |
|------------|--------|----------------------------------------|
| Training   | 50 000 | Estimate parameters φ                  |
| Validation | 10 000 | Monitor generalization during training |
| Test       | 10 000 | Final unbiased accuracy estimate       |

---
## 1  Imports

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import random

---
## 2  Load and Inspect MNIST Data

The `mnist_loader` module reads the `mnist.pkl.gz` file and returns three splits.

* **training_data** – list of 50 000 tuples `(x, y)` where `x` is a `(784, 1)` column
  vector and `y` is a `(10, 1)` one-hot label vector.
* **validation_data** / **test_data** – list of 10 000 tuples `(x, y)` where `y` is
  the integer digit label (0–9).

In [2]:
import mnist_loader
training_data, validation_data, test_data = mnist_loader.load_data_wrapper()

print(f"Training samples  : {len(training_data)}")
print(f"Validation samples: {len(validation_data)}")
print(f"Test samples      : {len(test_data)}")

ModuleNotFoundError: No module named 'mnist_loader'

In [ ]:
# ── Visualise the first five training images ──────────────────────────────────
num_images = 5
fig, axes = plt.subplots(1, num_images, figsize=(15, 3))

for i in range(num_images):
    x, y = training_data[i]
    image = np.reshape(x, (28, 28))
    label = np.argmax(y)       # one-hot → integer
    axes[i].imshow(image, cmap='gray')
    axes[i].set_title(f"Label: {label}", fontsize=14)
    axes[i].axis('off')

plt.suptitle("Sample training images", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 3  Network Definition

The `SimpleNN` class implements:

| Method | Description |
|--------|-------------|
| `__init__(sizes)` | Build the network; Gaussian weight/bias initialisation |
| `feedforward(a)` | Forward pass: compute the output vector for one input |
| `SGD(...)` | Mini-batch SGD training loop |
| `update_mini_batch(batch, lr)` | Accumulate gradients and apply one parameter update |
| `backprop(x, y)` | Backpropagation: return `(∇b, ∇w)` for a single example |
| `evaluate(data)` | Count correct predictions on `data` |
| `calculate_loss(data)` | Compute average MSE over `data` |

In [ ]:
def sigmoid(z):
    """Element-wise sigmoid activation: σ(z) = 1 / (1 + exp(−z))."""
    return 1.0 / (1.0 + np.exp(-z))

def sigmoid_prime(z):
    """Derivative of sigmoid: σ'(z) = σ(z)(1 − σ(z))."""
    return sigmoid(z) * (1 - sigmoid(z))


class SimpleNN:
    """
    Feedforward neural network with an arbitrary number of layers.

    Parameters
    ----------
    sizes : list of int
        Number of neurons in each layer (input → hidden(s) → output).
        Example: [784, 30, 10]  →  784-input, 30-hidden, 10-output network.
    """

    def __init__(self, sizes):
        self.num_layers = len(sizes)
        self.sizes = sizes
        # Gaussian initialisation: biases for layers 1…L, weights for each pair
        self.biases  = [np.random.randn(y, 1) for y in sizes[1:]]
        self.weights = [np.random.randn(y, x)
                        for x, y in zip(sizes[:-1], sizes[1:])]

    # ------------------------------------------------------------------
    # Forward pass
    # ------------------------------------------------------------------
    def feedforward(self, a):
        """Return the network output vector for input `a`."""
        for b, w in zip(self.biases, self.weights):
            a = sigmoid(np.dot(w, a) + b)
        return a

    # ------------------------------------------------------------------
    # SGD training loop
    # ------------------------------------------------------------------
    def SGD(self, training_data, epochs, mini_batch_size, learning_rate,
            validation_data=None):
        """
        Train with mini-batch SGD.

        Returns
        -------
        (training_loss_history, validation_acc_history)
        """
        training_loss_history  = []
        validation_acc_history = []

        n     = len(training_data)
        n_val = len(validation_data) if validation_data else 0

        for epoch in range(epochs):
            # Shuffle and partition into mini-batches
            np.random.shuffle(training_data)
            mini_batches = [
                training_data[k : k + mini_batch_size]
                for k in range(0, n, mini_batch_size)
            ]

            for mini_batch in mini_batches:
                self.update_mini_batch(mini_batch, learning_rate)

            # ── Record metrics ────────────────────────────────────────
            epoch_loss = self.calculate_loss(training_data)
            training_loss_history.append(epoch_loss)

            if validation_data:
                correct  = self.evaluate(validation_data)
                accuracy = correct / n_val
                validation_acc_history.append(accuracy)
                print(f"Epoch {epoch:>2d}: "
                      f"Val accuracy = {correct}/{n_val} "
                      f"({accuracy*100:.2f}%) | "
                      f"Train MSE = {epoch_loss:.4f}")
            else:
                print(f"Epoch {epoch:>2d} complete | Train MSE = {epoch_loss:.4f}")

        return training_loss_history, validation_acc_history

    # ------------------------------------------------------------------
    # Parameter update (one mini-batch)
    # ------------------------------------------------------------------
    def update_mini_batch(self, mini_batch, learning_rate):
        """
        Accumulate gradients over the mini-batch and apply the SGD update:
            φ ← φ − (ρ/B) Σ ∇φ ℓ_s(φ)
        """
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]

        for x, y in mini_batch:
            delta_nabla_b, delta_nabla_w = self.backprop(x, y)
            nabla_b = [nb + dnb for nb, dnb in zip(nabla_b, delta_nabla_b)]
            nabla_w = [nw + dnw for nw, dnw in zip(nabla_w, delta_nabla_w)]

        scale = learning_rate / len(mini_batch)
        self.weights = [w - scale * nw for w, nw in zip(self.weights, nabla_w)]
        self.biases  = [b - scale * nb for b, nb in zip(self.biases,  nabla_b)]

    # ------------------------------------------------------------------
    # Backpropagation
    # ------------------------------------------------------------------
    def backprop(self, x, y):
        """
        Compute (∇b, ∇w) for the MSE loss on a single training example (x, y)
        using the backpropagation algorithm.

        Forward pass
        ------------
        For each layer ℓ:
            z^(ℓ) = W^(ℓ) a^(ℓ−1) + b^(ℓ)      (pre-activation)
            a^(ℓ) = σ(z^(ℓ))                     (activation)

        Backward pass
        -------------
        Output layer:   δ^(L) = ∂ℒ/∂f · σ'(z^(L))
                               = 2(f(x;φ) − y) · σ'(z^(L))   [MSE]
        Hidden layer ℓ: δ^(ℓ) = (W^(ℓ+1)ᵀ δ^(ℓ+1)) ⊙ σ'(z^(ℓ))
        Weight gradient: ∂ℒ/∂W^(ℓ) = δ^(ℓ) (a^(ℓ−1))ᵀ
        Bias gradient:   ∂ℒ/∂b^(ℓ) = δ^(ℓ)
        """
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]

        # ── Forward pass ──────────────────────────────────────────────
        activation  = x
        activations = [x]   # stores a^(0), a^(1), …, a^(L)
        zs          = []    # stores z^(1), …, z^(L)

        for b, w in zip(self.biases, self.weights):
            z = np.dot(w, activation) + b
            zs.append(z)
            activation = sigmoid(z)
            activations.append(activation)

        # ── Backward pass ─────────────────────────────────────────────
        # Output-layer error signal: δ^(L) = 2(a^(L) − y) ⊙ σ'(z^(L))
        delta        = self.cost_derivative(activations[-1], y) * sigmoid_prime(zs[-1])
        nabla_b[-1]  = delta
        nabla_w[-1]  = np.dot(delta, activations[-2].T)

        # Propagate δ back through hidden layers
        for l in range(2, self.num_layers):
            z            = zs[-l]
            sp           = sigmoid_prime(z)
            delta        = np.dot(self.weights[-l + 1].T, delta) * sp
            nabla_b[-l]  = delta
            nabla_w[-l]  = np.dot(delta, activations[-l - 1].T)

        return nabla_b, nabla_w

    # ------------------------------------------------------------------
    # Evaluation helpers
    # ------------------------------------------------------------------
    def evaluate(self, test_data):
        """Return the number of correct predictions on `test_data`."""
        results = [(np.argmax(self.feedforward(x)), y) for x, y in test_data]
        return sum(int(pred == label) for pred, label in results)

    def cost_derivative(self, output_activations, y):
        """Gradient of the MSE loss w.r.t. the output activations: 2(a − y)."""
        return 2 * (output_activations - y)

    def calculate_loss(self, data):
        """Average MSE loss over all examples in `data`."""
        total = sum(
            0.5 * np.linalg.norm(self.feedforward(x) - y) ** 2
            for x, y in data
        )
        return total / len(data)

    def predict(self, x):
        """
        Return (predicted_class, output_vector) for a single input `x`.
        The output_vector contains the ten sigmoid activations, one per
        digit class.
        """
        output = self.feedforward(x)
        return int(np.argmax(output)), output.flatten()

---
## 4  Train the Network

In [ ]:
# ── Instantiate and train ─────────────────────────────────────────────────────
HIDDEN_UNITS   = 30
EPOCHS         = 30
MINI_BATCH_SIZE = 10
LEARNING_RATE  = 3.0

net = SimpleNN([784, HIDDEN_UNITS, 10])

train_loss, val_acc = net.SGD(
    training_data,
    epochs          = EPOCHS,
    mini_batch_size = MINI_BATCH_SIZE,
    learning_rate   = LEARNING_RATE,
    validation_data = validation_data
)

---
## 5  Learning Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Training loss
ax1.plot(range(EPOCHS), train_loss, color='crimson', marker='o',
         markersize=4, linewidth=1.5, label='Training MSE loss')
ax1.set_title('Training Loss over Epochs', fontsize=13)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('MSE loss')
ax1.grid(True, alpha=0.4)
ax1.legend()

# Validation accuracy
ax2.plot(range(EPOCHS), [v * 100 for v in val_acc], color='steelblue',
         marker='s', markersize=4, linewidth=1.5, label='Validation accuracy')
ax2.set_title('Validation Accuracy over Epochs', fontsize=13)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_ylim(0, 100)
ax2.grid(True, alpha=0.4)
ax2.legend()

plt.suptitle('Learning curves – MNIST digit classifier', fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nFinal validation accuracy: {val_acc[-1]*100:.2f}%")

---
## 6  Evaluation on the Held-Out Test Set

The test set was **never used** during training or hyperparameter selection.
Evaluating on it now gives an unbiased estimate of the network's
generalisation performance — i.e., an approximation of the
true risk $\mathcal{L}_{\text{true}}(\phi)$.

In [ ]:
n_test    = len(test_data)
n_correct = net.evaluate(test_data)
test_acc  = n_correct / n_test

print("=" * 50)
print(" TEST-SET EVALUATION (unseen data)")
print("=" * 50)
print(f" Correct predictions : {n_correct} / {n_test}")
print(f" Accuracy            : {test_acc * 100:.2f}%")
print(f" Errors              : {n_test - n_correct}")
print("=" * 50)

---
## 7  Visualise Correctly Classified Test Examples

We display a random selection of 10 correctly classified test images
together with the network's output vector (the 10 sigmoid activations).

In [ ]:
# Collect all correct predictions
correct_examples = [
    (x, y) for x, y in test_data
    if net.predict(x)[0] == y
]

# Draw 10 at random
sample = random.sample(correct_examples, 10)

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
axes = axes.flatten()

for ax, (x, true_label) in zip(axes, sample):
    pred_class, output_vec = net.predict(x)
    image = np.reshape(x, (28, 28))

    # Show digit image
    ax.imshow(image, cmap='gray')
    ax.set_title(f"True: {true_label} | Pred: {pred_class}",
                 fontsize=10, color='darkgreen')
    ax.axis('off')

    # Annotate with the winning activation value
    ax.text(0.5, -0.05,
            f"activation[{pred_class}] = {output_vec[pred_class]:.3f}",
            ha='center', transform=ax.transAxes,
            fontsize=8, color='navy')

plt.suptitle('Correctly Classified Test Samples', fontsize=14)
plt.tight_layout()
plt.show()

---
## 8  Visualise Misclassified Test Examples

Inspecting failure cases provides qualitative insight into which digits
the network finds ambiguous.  Common confusion pairs include **4 ↔ 9**,
**3 ↔ 5**, and **7 ↔ 1**.

In [ ]:
# Collect all misclassified examples
error_examples = [
    (x, y) for x, y in test_data
    if net.predict(x)[0] != y
]

print(f"Total errors on test set: {len(error_examples)}")

# Draw up to 10 errors at random
n_show  = min(10, len(error_examples))
sample  = random.sample(error_examples, n_show)

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
axes = axes.flatten()

for ax, (x, true_label) in zip(axes, sample):
    pred_class, output_vec = net.predict(x)
    image = np.reshape(x, (28, 28))

    ax.imshow(image, cmap='gray')
    ax.set_title(f"True: {true_label} | Pred: {pred_class}",
                 fontsize=10, color='crimson')
    ax.axis('off')

    # Show both the true-class and predicted-class activations
    ax.text(0.5, -0.05,
            f"act[{true_label}]={output_vec[true_label]:.2f}  "
            f"act[{pred_class}]={output_vec[pred_class]:.2f}",
            ha='center', transform=ax.transAxes,
            fontsize=7.5, color='saddlebrown')

plt.suptitle('Misclassified Test Samples', fontsize=14)
plt.tight_layout()
plt.show()

---
## 9  Output Activation Bar Chart for a Single Test Example

For one randomly chosen test image we plot the full 10-dim output vector.
The tallest bar corresponds to the network's prediction.

In [ ]:
# Pick a random test example
x_demo, y_demo = random.choice(test_data)
pred_class, output_vec = net.predict(x_demo)

fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(12, 4),
                                      gridspec_kw={'width_ratios': [1, 2.5]})

# Left: the digit image
ax_img.imshow(np.reshape(x_demo, (28, 28)), cmap='gray')
ax_img.set_title(f"True label: {y_demo}  |  Prediction: {pred_class}",
                 fontsize=12,
                 color='darkgreen' if pred_class == y_demo else 'crimson')
ax_img.axis('off')

# Right: output activations
colors = ['steelblue'] * 10
colors[pred_class] = 'crimson'     # highlight predicted class

ax_bar.bar(range(10), output_vec, color=colors, edgecolor='black', linewidth=0.5)
ax_bar.set_xticks(range(10))
ax_bar.set_xticklabels([str(d) for d in range(10)], fontsize=11)
ax_bar.set_ylim(0, 1.05)
ax_bar.set_xlabel('Digit class', fontsize=12)
ax_bar.set_ylabel('Output activation  σ(z_k)', fontsize=12)
ax_bar.set_title('Network output activations (all 10 classes)', fontsize=12)
ax_bar.axhline(y=0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
ax_bar.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nFull output vector:")
for d, act in enumerate(output_vec):
    marker = " ← predicted" if d == pred_class else ""
    print(f"  class {d}: {act:.4f}{marker}")

---
## 10  Confusion Matrix

The confusion matrix counts how many times the true digit $k$ was
predicted as digit $\hat{k}$.  Off-diagonal entries reveal systematic
confusion pairs.

In [ ]:
# Build 10 × 10 confusion matrix
C = np.zeros((10, 10), dtype=int)
for x, y_true in test_data:
    y_pred = net.predict(x)[0]
    C[y_true, y_pred] += 1

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(C, cmap='Blues')

# Annotate each cell
for i in range(10):
    for j in range(10):
        color = 'white' if C[i, j] > C.max() * 0.6 else 'black'
        ax.text(j, i, str(C[i, j]),
                ha='center', va='center', fontsize=9, color=color)

ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels([str(d) for d in range(10)])
ax.set_yticklabels([str(d) for d in range(10)])
ax.set_xlabel('Predicted digit', fontsize=12)
ax.set_ylabel('True digit', fontsize=12)
ax.set_title('Confusion Matrix — Test Set', fontsize=13)

plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

# Per-class accuracy
print("\nPer-class accuracy:")
for d in range(10):
    n_class   = C[d, :].sum()
    n_correct = C[d, d]
    print(f"  Digit {d}: {n_correct}/{n_class}  ({100*n_correct/n_class:.1f}%)")

---
## 11  Summary

| Metric | Value |
|--------|-------|
| Training MSE (final epoch) | see learning curves |
| Validation accuracy | ~95% |
| **Test accuracy (unseen data)** | **~95%** |

The near-identical validation and test accuracies confirm that the network
generalises well and has not been over-fitted to the training data.

### Possible improvements
| Change | Expected benefit |
|--------|------------------|
| Replace sigmoid hidden activation with **ReLU** | Faster training, avoids vanishing gradients |
| Replace MSE loss with **cross-entropy** | Better calibrated probabilities; more natural for classification |
| Add **dropout** regularisation | Reduce overfitting at larger widths |
| Increase hidden units (e.g. 100–300) | Higher capacity |
| Use a **convolutional** architecture | Exploit spatial structure of images; state-of-the-art < 0.3% error |